# SparseGPT — facebook/opt-125m | wikitext2 | All 3 Methods

In [ ]:
!pip install -q transformers==4.36.2 tokenizers==0.15.2 datasets==2.18.0 huggingface_hub==0.23.4 --only-binary=:all:
!pip install -q sentencepiece accelerate
print('Done')

In [ ]:
import os
if not os.path.exists('sparsegpt'):
    !git clone https://github.com/IST-DASLab/sparsegpt
%cd sparsegpt
!mkdir -p sparse_sparsegpt sparse_magnitude sparse_movement

## Imports & Setup

In [ ]:
import torch
import torch.nn.utils.prune as prune
import time
import subprocess
import re
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL     = 'facebook/opt-125m'
DATASET   = 'wikitext2'
SPARSITY_LEVELS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
device    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device : {device}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Helper — Inference Metrics (Memory, Latency, Throughput, Energy)

In [ ]:
def measure_inference(model, tokenizer, num_runs=5):
    model.eval()
    if device == 'cuda':
        torch.cuda.synchronize()
        memory_mb = round(torch.cuda.memory_allocated() / 1e6, 1)
    else:
        import psutil
        memory_mb = round(__import__('psutil').Process(os.getpid()).memory_info().rss / 1e6, 1)

    inputs  = tokenizer('The quick brown fox', return_tensors='pt').to(device)
    MAX_NEW = 50

    with torch.no_grad():
        _ = model.generate(inputs['input_ids'], max_new_tokens=5, do_sample=False)

    latencies = []
    for _ in range(num_runs):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            _ = model.generate(inputs['input_ids'], max_new_tokens=MAX_NEW, do_sample=False)
        if device == 'cuda': torch.cuda.synchronize()
        latencies.append(time.time() - t0)

    avg_lat    = float(np.mean(latencies))
    throughput = round(MAX_NEW / avg_lat, 2)
    tdp_w      = 400 if 'a100' in torch.cuda.get_device_name(0).lower() else 70
    energy_j   = round(tdp_w * avg_lat, 3)

    return {
        'memory_mb':  memory_mb,
        'latency_s':  round(avg_lat, 3),
        'throughput': throughput,
        'energy_j':   energy_j,
    }

## Method 1 — SparseGPT

Uses the official `opt.py` script. Perplexity is taken directly from script output.

In [ ]:
results_sparsegpt = {}

for sparsity in SPARSITY_LEVELS:
    spct      = int(sparsity * 100)
    save_path = f'sparse_sparsegpt/s{spct}'

    print(f'\nSparseGPT {spct}%')

    t0     = time.time()
    result = subprocess.run(
        ['python', 'opt.py', MODEL, DATASET,
         '--sparsity', str(sparsity), '--save', save_path],
        capture_output=True, text=True
    )
    pruning_time = round(time.time() - t0, 1)

    output = result.stdout + result.stderr
    match  = re.search(r'wikitext2.*?Perplexity:\s*([\d.]+)', output,
                       re.IGNORECASE | re.DOTALL)
    if not match:
        matches = re.findall(r'Perplexity:\s*([\d.]+)', output)
        ppl = float(matches[0]) if matches else None
    else:
        ppl = float(match.group(1))

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model     = AutoModelForCausalLM.from_pretrained(
                    save_path, torch_dtype=torch.float16).to(device)
    inf = measure_inference(model, tokenizer)
    del model
    torch.cuda.empty_cache()

    results_sparsegpt[spct] = {
        'sparsity':      f'{spct}%',
        'perplexity':    ppl,
        'memory_mb':     inf['memory_mb'],
        'latency_s':     inf['latency_s'],
        'pruning_time':  pruning_time,
        'energy_j':      inf['energy_j'],
        'throughput':    inf['throughput'],
    }

    print(f'  PPL={ppl} | Mem={inf["memory_mb"]}MB | Lat={inf["latency_s"]}s | '
          f'Tok/s={inf["throughput"]} | PruneTime={pruning_time}s | Energy={inf["energy_j"]}J')

print('\nSparseGPT done')
print(f'\n{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
print('-'*75)
for s, r in results_sparsegpt.items():
    print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | {str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | {str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')

## Method 2 — Magnitude Pruning

Uses `torch.nn.utils.prune.l1_unstructured`. Perplexity computed via `opt.py` by saving and re-evaluating the pruned model.

In [ ]:
results_magnitude = {}

for sparsity in SPARSITY_LEVELS:
    spct      = int(sparsity * 100)
    save_path = f'sparse_magnitude/s{spct}'
    os.makedirs(save_path, exist_ok=True)

    print(f'\nMagnitude {spct}%')

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL, torch_dtype=torch.float16).to(device)

    t0 = time.time()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=sparsity)
            prune.remove(module, 'weight')
    pruning_time = round(time.time() - t0, 1)

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    inf = measure_inference(model, tokenizer)
    del model
    torch.cuda.empty_cache()

    result = subprocess.run(
        ['python', 'opt.py', save_path, DATASET, '--sparsity', '0'],
        capture_output=True, text=True
    )
    output = result.stdout + result.stderr
    match  = re.search(r'wikitext2.*?Perplexity:\s*([\d.]+)', output,
                       re.IGNORECASE | re.DOTALL)
    if not match:
        matches = re.findall(r'Perplexity:\s*([\d.]+)', output)
        ppl = float(matches[0]) if matches else None
    else:
        ppl = float(match.group(1))

    results_magnitude[spct] = {
        'sparsity':      f'{spct}%',
        'perplexity':    ppl,
        'memory_mb':     inf['memory_mb'],
        'latency_s':     inf['latency_s'],
        'pruning_time':  pruning_time,
        'energy_j':      inf['energy_j'],
        'throughput':    inf['throughput'],
    }

    print(f'  PPL={ppl} | Mem={inf["memory_mb"]}MB | Lat={inf["latency_s"]}s | '
          f'Tok/s={inf["throughput"]} | PruneTime={pruning_time}s | Energy={inf["energy_j"]}J')

print('\nMagnitude done')
print(f'\n{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
print('-'*75)
for s, r in results_magnitude.items():
    print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | {str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | {str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')

## Method 3 — Movement Pruning

Scores weights by `|weight × gradient|` using a calibration pass. Saves and evaluates with `opt.py`.

In [ ]:
results_movement = {}

for sparsity in SPARSITY_LEVELS:
    spct      = int(sparsity * 100)
    save_path = f'sparse_movement/s{spct}'
    os.makedirs(save_path, exist_ok=True)

    print(f'\nMovement {spct}%')

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL, torch_dtype=torch.float32).to(device)
    model.train()

    scores = {n: torch.zeros_like(m.weight.data)
              for n, m in model.named_modules()
              if isinstance(m, torch.nn.Linear)}

    calib = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    texts = [x for x in calib['text'] if len(x.strip()) > 50][:16]

    t0 = time.time()
    for text in texts:
        enc = tokenizer(text, return_tensors='pt',
                        truncation=True, max_length=128).to(device)
        out = model(**enc, labels=enc['input_ids'])
        out.loss.backward()
        for n, m in model.named_modules():
            if isinstance(m, torch.nn.Linear) and m.weight.grad is not None:
                scores[n] += (m.weight.data * m.weight.grad).abs()
        model.zero_grad()

    model.eval()
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear):
            k      = int(sparsity * scores[n].numel())
            thresh = scores[n].flatten().kthvalue(k).values
            m.weight.data *= (scores[n] > thresh).float()
    pruning_time = round(time.time() - t0, 1)

    model = model.half()
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    inf = measure_inference(model, tokenizer)
    del model
    torch.cuda.empty_cache()

    result = subprocess.run(
        ['python', 'opt.py', save_path, DATASET, '--sparsity', '0'],
        capture_output=True, text=True
    )
    output = result.stdout + result.stderr
    match  = re.search(r'wikitext2.*?Perplexity:\s*([\d.]+)', output,
                       re.IGNORECASE | re.DOTALL)
    if not match:
        matches = re.findall(r'Perplexity:\s*([\d.]+)', output)
        ppl = float(matches[0]) if matches else None
    else:
        ppl = float(match.group(1))

    results_movement[spct] = {
        'sparsity':      f'{spct}%',
        'perplexity':    ppl,
        'memory_mb':     inf['memory_mb'],
        'latency_s':     inf['latency_s'],
        'pruning_time':  pruning_time,
        'energy_j':      inf['energy_j'],
        'throughput':    inf['throughput'],
    }

    print(f'  PPL={ppl} | Mem={inf["memory_mb"]}MB | Lat={inf["latency_s"]}s | '
          f'Tok/s={inf["throughput"]} | PruneTime={pruning_time}s | Energy={inf["energy_j"]}J')

print('\nMovement done')
print(f'\n{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
print('-'*75)
for s, r in results_movement.items():
    print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | {str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | {str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')

## Final Summary — All 3 Methods

In [ ]:
all_results = {
    'SparseGPT': results_sparsegpt,
    'Magnitude': results_magnitude,
    'Movement':  results_movement,
}

for method, res in all_results.items():
    print(f'\n=== {method} ===')
    print(f'{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
    print('-'*75)
    for s, r in res.items():
        print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | '
              f'{str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | '
              f'{str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')

print('\n=== Perplexity Comparison (wikitext2) ===')
print(f'{"Sparsity":>10} | {"SparseGPT":>11} | {"Magnitude":>11} | {"Movement":>11}')
print('-'*50)
for s in [20,30,40,50,60,70]:
    sg = results_sparsegpt.get(s,{}).get('perplexity','N/A')
    mg = results_magnitude.get(s,{}).get('perplexity','N/A')
    mv = results_movement.get(s,{}).get('perplexity','N/A')
    print(f'{str(s)+"%":>10} | {str(sg):>11} | {str(mg):>11} | {str(mv):>11}')